In [0]:
%python
from pyspark.sql import functions as F

L  = 'northbridge.lakeflow'
DQ = 'northbridge.ops.dq_results'
t  = lambda name: spark.table(f'{L}.{name}')

n_bronze = t('bronze_transactions').count()
n_silver = t('silver_transactions').count()
n_rej    = t('silver_transactions_rejected').count()
dim      = t('dim_customer')
n_cur    = dim.filter('`__END_AT` IS NULL').count()
multi    = dim.filter('`__END_AT` IS NULL').groupBy('customer_id').count().filter('count > 1').count()
n_fact   = t('gold_fact_transactions').count()
n_c360   = t('gold_customer_360').count()

checks = [
  ('lf_bronze_eq_silver_plus_rejected', n_bronze == n_silver + n_rej, n_bronze),
  ('lf_silver_no_duplicate_ids', n_silver == t('silver_transactions').select('txn_id').distinct().count(), n_silver),
  ('lf_dim_one_current_per_customer', multi == 0, multi),
  ('lf_fact_eq_silver', n_fact == n_silver, n_fact),
  ('lf_customer360_eq_current_dim', n_c360 == n_cur, n_c360),
]

rows = [(n, 'PASS' if ok else 'FAIL', int(v), '') for n, ok, v in checks]
out = spark.createDataFrame(rows, 'check_name string, status string, row_count long, notes string')
(out.withColumn('run_ts', F.current_timestamp())
    .select('check_name', 'run_ts', 'status', 'row_count', 'notes')
    .write.mode('append').saveAsTable(DQ))
display(out)

failed = [n for n, ok, _ in checks if not ok]
if failed:
    raise Exception(f'Data quality checks failed: {failed}')